In [1]:
from pathlib import Path
import json

PROJECT_DIR = Path.cwd()

# Handle cases where Jupyter starts in a different directory
if not (PROJECT_DIR / "data" / "raw").exists():
    PROJECT_DIR = Path.home() / "Documents" / "Biomedical-RAG-Research"

DATA_PATH = PROJECT_DIR / "data" / "raw" / "ori_pqal.json"

print("Project directory:", PROJECT_DIR)
print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)

Project directory: /Users/shubhrasharma/Documents/Biomedical-RAG-Research
Dataset exists: True
Dataset path: /Users/shubhrasharma/Documents/Biomedical-RAG-Research/data/raw/ori_pqal.json


In [2]:
with open(DATA_PATH, "r", encoding="utf-8") as file:
    data = json.load(file)

print("Data type:", type(data).__name__)
print("Number of records:", len(data))

# Inspect one record without printing the entire dataset
if isinstance(data, dict):
    first_id = next(iter(data))
    print("First record ID:", first_id)
    print("Fields:", list(data[first_id].keys()))
    print("Example question:", data[first_id].get("QUESTION"))
    print("Gold label:", data[first_id].get("final_decision"))

Data type: dict
Number of records: 1000
First record ID: 21645374
Fields: ['QUESTION', 'CONTEXTS', 'LABELS', 'MESHES', 'YEAR', 'reasoning_required_pred', 'reasoning_free_pred', 'final_decision', 'LONG_ANSWER']
Example question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Gold label: yes


In [3]:
from collections import Counter
import json

# 1. Count the gold labels
labels = [
    record["final_decision"]
    for record in data.values()
]

print("Gold label distribution:")
for label, count in sorted(Counter(labels).items()):
    print(f"{label}: {count}")

# 2. Inspect the structure of CONTEXTS
first_record = next(iter(data.values()))

print("\nCONTEXTS type:", type(first_record["CONTEXTS"]).__name__)
print("Number of contexts in first record:", len(first_record["CONTEXTS"]))

if first_record["CONTEXTS"]:
    print("\nFirst context preview:")
    print(str(first_record["CONTEXTS"][0])[:500])

# 3. Inspect the existing test ground-truth file
test_path = PROJECT_DIR / "pubmedqa-source" / "data" / "test_ground_truth.json"

print("\nTest ground-truth file exists:", test_path.exists())

if test_path.exists():
    with open(test_path, "r", encoding="utf-8") as file:
        test_data = json.load(file)

    print("Test file type:", type(test_data).__name__)
    print("Number of entries:", len(test_data))

    if isinstance(test_data, dict):
        print("First test entry:", next(iter(test_data.items())))
    elif isinstance(test_data, list) and test_data:
        print("First test entry:", test_data[0])

Gold label distribution:
maybe: 110
no: 338
yes: 552

CONTEXTS type: list
Number of contexts in first record: 2

First context preview:
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has b

Test ground-truth file exists: True
Test file type: dict
Number of entries: 500
First test entry: ('12377809', 'yes')


In [4]:
import json

test_path = PROJECT_DIR / "pubmedqa-source" / "data" / "test_ground_truth.json"

with open(test_path, "r", encoding="utf-8") as file:
    test_labels = json.load(file)

# Verify that every test ID exists in the labelled dataset
missing_ids = set(test_labels) - set(data)

# Verify that test labels match the full dataset
label_mismatches = [
    pmid
    for pmid, label in test_labels.items()
    if pmid in data and data[pmid]["final_decision"] != label
]

print("Test questions:", len(test_labels))
print("IDs missing from dataset:", len(missing_ids))
print("Label mismatches:", len(label_mismatches))

print("\nTest label distribution:")
from collections import Counter
print(dict(Counter(test_labels.values())))

Test questions: 500
IDs missing from dataset: 0
Label mismatches: 0

Test label distribution:
{'yes': 276, 'no': 169, 'maybe': 55}


In [5]:
import json

evaluation_dir = PROJECT_DIR / "data" / "evaluation"
evaluation_dir.mkdir(parents=True, exist_ok=True)

# Save only the questions and IDs for model input
test_questions = {
    pmid: {
        "question": data[pmid]["QUESTION"]
    }
    for pmid in test_labels
}

with open(evaluation_dir / "test_questions.json", "w", encoding="utf-8") as file:
    json.dump(test_questions, file, indent=2, ensure_ascii=False)

# Store gold labels separately for evaluation only
with open(evaluation_dir / "test_labels.json", "w", encoding="utf-8") as file:
    json.dump(test_labels, file, indent=2, ensure_ascii=False)

print("Saved test questions:", len(test_questions))
print("Saved test labels:", len(test_labels))
print("Evaluation directory:", evaluation_dir)

Saved test questions: 500
Saved test labels: 500
Evaluation directory: /Users/shubhrasharma/Documents/Biomedical-RAG-Research/data/evaluation
